# LightGBM khởi phát với data cleaning, quét ngưỡng và baseline tương đối

Input là smoke_detection_iot_transient_v1.csv. Notebook xây đặc trưng trên toàn chuỗi trước khi chọn hàng train. Các đoạn tiếp diễn/hồi phục bị loại khỏi loss và validation nhị phân; chúng được giữ trong nguồn để cung cấp lịch sử cảm biến.

Luồng dữ liệu khi chạy là CSV v1 đầy đủ (62.630 hàng) → tính lịch sử và đặc trưng → chọn clean_mask → tập học v2 sạch (14.864 hàng). Tên file v1 chỉ phiên bản nguồn đầu vào. CSV smoke_transient_clean_features_v2.csv là bản đặc trưng sạch được notebook xuất sau xử lý. Khi dùng notebook này trên Kaggle, thêm CSV v1 qua Add Input.

So sánh ba cấu hình trên cùng các hàng validation sạch. source13 là đối chứng train theo nhãn v1 cũ, clean13 chỉ học nền được chọn và khởi phát, clean15 thêm hai đặc trưng tương đối. Các cặp phiên 1–5 và 2–3 nằm cùng nhóm.

Nhãn khởi phát hiện là chú giải suy ra từ cảm biến. Các mốc nền sạch của phiên 1/5 là lựa chọn thủ công từ đồ thị. Kết quả cần được hiểu theo phạm vi đó.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import zipfile

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None
    display = print

DATASET_PATH = None
OUTPUT_DIR = Path('/kaggle/working/transient_clean_hybrid_v2') if Path('/kaggle/working').exists() else Path('output/analysis/transient_clean_hybrid_v2')
BASELINE_SAMPLES = 10
TARGET_RECALL = 0.80
THRESHOLDS = np.round(np.arange(0.05, 0.251, 0.01), 2)
FINAL_VARIANT = 'clean13'
TARGET = 'Fire_Alarm_Transient'
FEATURES13 = [
    'd_temp_1', 'd_temp_5', 'd_hum_1', 'd_hum_5',
    'd_pm1_1', 'd_pm1_5', 'd_pm25_1', 'd_pm25_5',
    'temp_roll_5_std', 'hum_roll_5_std', 'pm1_roll_5_std',
    'pm25_roll_5_std', 'pm_ratio',
]
FEATURES15 = FEATURES13 + ['rel_temp', 'rel_pm25']
SENSORS = {'Temperature[C]': 'temp', 'Humidity[%]': 'hum', 'PM1.0': 'pm1', 'PM2.5': 'pm25'}
PARAMETERS = dict(n_estimators=150, max_depth=6, num_leaves=31,
                  learning_rate=0.05, random_state=42, n_jobs=2, verbosity=-1)
NORMAL_END_CNT = {1: 3900, 5: 3900, 2: 3172, 3: 3172, 4: 15}
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('Model parameters:', PARAMETERS)


Model parameters: {'n_estimators': 150, 'max_depth': 6, 'num_leaves': 31, 'learning_rate': 0.05, 'random_state': 42, 'n_jobs': 2, 'verbosity': -1}


## Baseline và vùng giữ lại

Baseline nhiệt và PM2.5 là trung vị 10 mẫu đầu liên tục. Chỉ dùng đặc trưng tương đối từ mẫu thứ 11, khi baseline đã được thu đủ. Hai giá trị được đóng băng cho cả chuỗi, reset khi khởi động/chuỗi đo gián đoạn. Median giúp giảm ảnh hưởng một mẫu lệch.

Phiên 4 khởi phát từ CNT=16 nên 30 mẫu đầu không phù hợp để lấy nền. Đoạn nền 10 mẫu hiện tại được kiểm tra với chú giải. Khi vận hành, cần một giai đoạn hiệu chuẩn nền mà người vận hành biết là sạch.

Giữ nhãn 0 trước mốc NORMAL_END_CNT của mỗi phiên và nhãn 1 ở các cửa sổ đã chú giải. Phiên 1/5 được dừng phần nền ở CNT=3900 để loại vùng nhiệt/ẩm đã thay đổi trước đợt PM. Những mẫu còn lại không được dùng làm âm trong clean13/clean15. Các đợt khởi phát sau vẫn được giữ là dương.


In [2]:
filename = 'smoke_detection_iot_transient_v1.csv'
if DATASET_PATH is None:
    paths = list(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
    for parent in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        paths.extend([parent / filename, parent / 'Model/smoke_transient_v1' / filename])
    paths = sorted({path.resolve() for path in paths if path.is_file()})
    if len(paths) != 1:
        raise ValueError(f'Set DATASET_PATH explicitly. Found {paths}')
    dataset_path = paths[0]
else:
    dataset_path = Path(DATASET_PATH)

df = pd.read_csv(dataset_path, float_precision='round_trip')
required = [*SENSORS, 'UTC', 'CNT', 'session_id', 'source_group', TARGET, 'training_eligible']
if not set(required).issubset(df):
    raise ValueError('Input must be the annotated v1 dataset')
if not df[TARGET].isin([0, 1]).all() or not np.isfinite(df[[*SENSORS, 'UTC', 'CNT']].to_numpy()).all():
    raise ValueError('Invalid labels or sensor measurements')

def boolean_column(series):
    values = series.astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False})
    if values.isna().any():
        raise ValueError(f'Invalid boolean column {series.name}')
    return values.astype(bool)

df['training_eligible'] = boolean_column(df['training_eligible'])
df = df.sort_values(['session_id', 'UTC', 'CNT']).reset_index(drop=True)
if not df.session_id.map({1: 'A', 5: 'A', 2: 'B', 3: 'B', 4: 'C'}).eq(df.source_group).all():
    raise ValueError('Invalid source grouping')
if df.duplicated(['session_id', 'CNT']).any():
    raise ValueError('Duplicate sample keys')

reset = df.session_id.ne(df.session_id.shift()) | df.UTC.diff().ne(1) | df.CNT.diff().ne(1)
block = reset.cumsum()
position = df.groupby(block, sort=False).cumcount()
normal_candidate = df.CNT.le(df.session_id.map(NORMAL_END_CNT)) & df[TARGET].eq(0)
X = pd.DataFrame(index=df.index)
for sensor, prefix in SENSORS.items():
    grouped = df[sensor].groupby(block, sort=False)
    for lag in (1, 5):
        X[f'd_{prefix}_{lag}'] = grouped.diff(lag).fillna(0.0)
    X[f'{prefix}_roll_5_std'] = grouped.transform(lambda part: part.rolling(5, min_periods=1).std(ddof=1).fillna(0.0))
X['pm_ratio'] = (df['PM2.5'] + 0.001) / (df['PM1.0'] + 0.001)
baseline_temp = pd.Series(np.nan, index=df.index)
baseline_pm25 = pd.Series(np.nan, index=df.index)
baseline_rows = []
for block_id, group in df.groupby(block, sort=False):
    if len(group) < BASELINE_SAMPLES:
        continue
    reference = group.head(BASELINE_SAMPLES)
    if not normal_candidate.loc[reference.index].all():
        raise ValueError('Baseline overlaps a non-background interval; require a clean calibration sequence')
    t0 = float(reference['Temperature[C]'].median())
    p0 = float(reference['PM2.5'].median())
    baseline_temp.loc[group.index] = t0
    baseline_pm25.loc[group.index] = p0
    baseline_rows.append({'session_id': int(group.session_id.iloc[0]), 'block': int(block_id),
                          'reference_start_cnt': int(reference.CNT.iloc[0]),
                          'reference_end_cnt': int(reference.CNT.iloc[-1]),
                          'temperature_baseline': t0, 'pm25_baseline': p0})
X['rel_temp'] = df['Temperature[C]'] - baseline_temp
X['rel_pm25'] = df['PM2.5'] - baseline_pm25
X = X.loc[:, FEATURES15]
ready = position.ge(max(5, BASELINE_SAMPLES)) & X.notna().all(axis=1)
v1_mask = df.training_eligible & ready
clean_mask = v1_mask & (normal_candidate | df[TARGET].eq(1))
y = df[TARGET].astype(int)
source_groups = df.source_group
if not np.isfinite(X.loc[v1_mask]).all().all():
    raise ValueError('Invalid engineered features')

cleaning_phase = pd.Series('excluded_continuation_or_unverified', index=df.index)
cleaning_phase.loc[normal_candidate] = 'normal'
cleaning_phase.loc[y.eq(1)] = 'onset'
cleaning_phase.loc[~df.training_eligible] = 'excluded_history_or_review'
cleaning_phase.loc[~ready] = 'baseline_warmup'
assert clean_mask[y.eq(1)].all(), 'Unexpected loss of annotated onset samples'
baseline_table = pd.DataFrame(baseline_rows)
display(baseline_table)
clean_table = df.loc[clean_mask].groupby('source_group')[TARGET].agg(rows='size', positive='sum')
clean_table['negative'] = clean_table['rows'] - clean_table['positive']
display(clean_table)
print('Retained:', int(clean_mask.sum()), '| omitted:', int((~clean_mask).sum()))
print('SOURCE INPUT:', dataset_path.name, '| rows:', len(df))
print('CLEAN TRAINING:', int(clean_mask.sum()), 'rows | positives:', int(y.loc[clean_mask].sum()),
      '| negatives:', int(y.loc[clean_mask].eq(0).sum()))
print('FINAL VARIANT:', FINAL_VARIANT, '| uses cleaned rows:', FINAL_VARIANT in ('clean13', 'clean15'))


   session_id  block  reference_start_cnt  reference_end_cnt  temperature_baseline  pm25_baseline
0           1      1                    0                  9                27.245          2.365
1           2      2                    0                  9                20.066          0.000
2           3      3                    0                  9                12.845          0.000
3           4      4                    0                  9                20.570          2.245
4           5      5                    0                  9                20.285          2.365
              rows  positive  negative
source_group                          
A             8440       658      7782
B             6352       108      6244
C               72        66         6
Retained: 14864 | omitted: 47766
SOURCE INPUT: smoke_detection_iot_transient_v1.csv | rows: 62630
CLEAN TRAINING: 14864 rows | positives: 832 | negatives: 14032
FINAL VARIANT: clean13 | uses cleaned rows: True


## Hàm chấm điểm và chọn ngưỡng

Ngưỡng được quét từ 0,05 đến 0,25, bước 0,01. Nếu có ngưỡng đạt recall ít nhất 80%, chọn ngưỡng có F1 cao nhất trong nhóm đó. Nếu không đạt, chọn F1 tốt nhất trong dải và báo rõ recall cao nhất có thể đạt trong dải.

Quét trực tiếp điểm OOF cho bảng phát triển. Để kiểm tra ngưỡng trên nhóm chưa được dùng chọn ngưỡng, clean13/clean15 còn chạy validation lồng nhau: với mỗi nhóm ngoài, chọn ngưỡng chỉ từ điểm ngoài fold của hai nhóm train, rồi chấm nhóm ngoài bằng ngưỡng đó.


In [3]:
def score_metrics(truth, probability):
    return {'rows': len(truth), 'positive': int(np.sum(truth)),
            'positive_rate': float(np.mean(truth)),
            'roc_auc': float(roc_auc_score(truth, probability)),
            'average_precision': float(average_precision_score(truth, probability)),
            'log_loss': float(log_loss(truth, probability, labels=[0, 1])),
            'brier_score': float(brier_score_loss(truth, probability))}


def prediction_metrics(truth, predicted):
    tn, fp, fn, tp = map(int, confusion_matrix(truth, predicted, labels=[0, 1]).ravel())
    return {'accuracy': float(accuracy_score(truth, predicted)),
            'precision': float(precision_score(truth, predicted, zero_division=0)),
            'recall': float(recall_score(truth, predicted, zero_division=0)),
            'f1': float(f1_score(truth, predicted, zero_division=0)),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}


def threshold_table(truth, scores):
    return pd.DataFrame([{'threshold': float(t), **prediction_metrics(truth, np.asarray(scores) >= t)} for t in THRESHOLDS])


def choose_threshold(table):
    feasible = table.loc[table.recall.ge(TARGET_RECALL)]
    pool = feasible if len(feasible) else table
    best = pool.sort_values(['f1', 'precision', 'threshold'], ascending=False).iloc[0]
    return {'threshold': float(best.threshold), 'recall_target_met': bool(len(feasible)),
            'maximum_recall_in_range': float(table.recall.max()),
            'selected_precision': float(best.precision), 'selected_recall': float(best.recall),
            'selected_f1': float(best.f1)}


def fit_model(mask, feature_names):
    if y.loc[mask].nunique() != 2:
        raise ValueError('A training partition lacks a class')
    estimator = lgb.LGBMClassifier(**PARAMETERS)
    estimator.fit(X.loc[mask, feature_names], y.loc[mask])
    return estimator


def probability(estimator, mask, feature_names):
    return estimator.predict_proba(X.loc[mask, feature_names])[:, list(estimator.classes_).index(1)]


def display_matrix(result, name):
    print(name)
    display(pd.DataFrame([[result['TN'], result['FP']], [result['FN'], result['TP']]],
                         index=['Thực tế 0', 'Thực tế 1'], columns=['Dự đoán 0', 'Dự đoán 1']))


In [4]:
variants = {'source13': (v1_mask, FEATURES13), 'clean13': (clean_mask, FEATURES13), 'clean15': (clean_mask, FEATURES15)}
experiments = {}
comparison_rows = []
per_group_rows = []
for variant, (train_mask, feature_names) in variants.items():
    oof = pd.Series(np.nan, index=df.index[clean_mask])
    nested_prediction = pd.Series(False, index=oof.index)
    nested_threshold_rows = []
    for held in sorted(source_groups.unique()):
        outer_train = train_mask & source_groups.ne(held)
        outer_val = clean_mask & source_groups.eq(held)
        estimator = fit_model(outer_train, feature_names)
        p = probability(estimator, outer_val, feature_names)
        oof.loc[outer_val[outer_val].index] = p
        per_group_rows.append({'variant': variant, 'held_out_group': held,
                               **score_metrics(y.loc[outer_val], p),
                               **prediction_metrics(y.loc[outer_val], p >= 0.5)})
        if variant != 'source13':
            calibration_mask = clean_mask & source_groups.ne(held)
            inner_scores = pd.Series(np.nan, index=df.index[calibration_mask])
            for inner_held in sorted(source_groups.loc[calibration_mask].unique()):
                inner_train = train_mask & source_groups.ne(held) & source_groups.ne(inner_held)
                inner_val = clean_mask & source_groups.eq(inner_held)
                assert not (inner_train & outer_val).any()
                inner_model = fit_model(inner_train, feature_names)
                inner_scores.loc[inner_val[inner_val].index] = probability(inner_model, inner_val, feature_names)
            assert inner_scores.notna().all()
            selected = choose_threshold(threshold_table(y.loc[inner_scores.index], inner_scores))
            prediction = p >= selected['threshold']
            nested_prediction.loc[outer_val[outer_val].index] = prediction
            nested_threshold_rows.append({'held_out_group': held, **selected,
                                          **prediction_metrics(y.loc[outer_val], prediction)})
        print(f'Completed {variant}, held-out group {held}', flush=True)
    assert oof.notna().all()
    sweep = threshold_table(y.loc[oof.index], oof)
    chosen = choose_threshold(sweep)
    development_prediction = oof.ge(chosen['threshold'])
    nested_summary = prediction_metrics(y.loc[oof.index], nested_prediction) if variant != 'source13' else None
    result = {'score_metrics': score_metrics(y.loc[oof.index], oof),
              'at_0_50': prediction_metrics(y.loc[oof.index], oof.ge(0.5)),
              'development_selection': chosen,
              'development_classification': prediction_metrics(y.loc[oof.index], development_prediction),
              'nested_classification': nested_summary, 'nested_thresholds': nested_threshold_rows}
    experiments[variant] = {'result': result, 'oof': oof, 'sweep': sweep, 'nested_prediction': nested_prediction}
    comparison_rows.append({'variant': variant, 'features': len(feature_names),
                            **result['score_metrics'], **chosen,
                            **result['development_classification']})

comparison_table = pd.DataFrame(comparison_rows)
group_metrics = pd.DataFrame(per_group_rows)
print('Development comparison: same cleaned validation rows for every variant')
display(comparison_table.round(6))
reference_table = pd.DataFrame([
    {'variant': name, 'threshold': 0.50, **experiment['result']['at_0_50']}
    for name, experiment in experiments.items()
])
print('Reference at 0.50, shown to check whether lowering the threshold helped')
display(reference_table.round(6))
print('Per-group metrics; classification columns below use 0.50 as a reference')
display(group_metrics.round(6))
for name in ['clean13', 'clean15']:
    print(f'{name}: nested threshold selection, held-out-group classification')
    display(pd.DataFrame(experiments[name]['result']['nested_thresholds']).round(6))
    display(pd.DataFrame([experiments[name]['result']['nested_classification']]).round(6))


Completed source13, held-out group A
Completed source13, held-out group B
Completed source13, held-out group C
Completed clean13, held-out group A
Completed clean13, held-out group B
Completed clean13, held-out group C
Completed clean15, held-out group A
Completed clean15, held-out group B
Completed clean15, held-out group C
Development comparison: same cleaned validation rows for every variant
    variant  features   rows  positive  positive_rate   roc_auc  average_precision  log_loss  brier_score  threshold  recall_target_met  maximum_recall_in_range  selected_precision  selected_recall  \
0  source13        13  14864       832       0.055974  0.758084           0.439206  0.386013     0.048571       0.06              False                 0.195913            0.848958         0.195913   
1   clean13        13  14864       832       0.055974  0.949329           0.905897  0.085395     0.016224       0.25               True                 0.908654            0.698706         0.908654   

## Ngưỡng phát triển và kết quả model cuối

Bảng sweep dưới đây được chọn trên validation OOF. Kết quả phát triển tại ngưỡng đó đã dùng nhãn OOF để chọn ngưỡng. Bảng nested phía trên chấm mỗi nhóm bằng ngưỡng được chọn hoàn toàn từ hai nhóm còn lại; hiệu năng có thể khác nhiều.

FINAL_VARIANT mặc định clean13 sau phép so sánh phát triển vì cân bằng precision/recall tốt hơn clean15. Cả hai cấu hình đều được train và validation trong notebook. Có thể chọn clean15 để xuất model 15 đặc trưng. Chỉ số đánh giá được báo trước khi fit lại trên toàn dữ liệu sạch.


In [5]:
selected_experiment = experiments[FINAL_VARIANT]
display(selected_experiment['sweep'].round(6))
print('Development threshold:', selected_experiment['result']['development_selection'])
display_matrix(selected_experiment['result']['development_classification'], 'OOF development confusion matrix')
display_matrix(selected_experiment['result']['nested_classification'], 'Nested held-out-group confusion matrix')
final_mask, final_features = variants[FINAL_VARIANT]
final_model = fit_model(final_mask, final_features)
final_probability = probability(final_model, final_mask, final_features)
final_threshold = selected_experiment['result']['development_selection']['threshold']
final_train_metrics = {**score_metrics(y.loc[final_mask], final_probability),
                       **prediction_metrics(y.loc[final_mask], final_probability >= final_threshold)}
print('Final model train metrics at development-selected threshold')
display(pd.DataFrame([final_train_metrics]).round(6))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audit = df[['source_row_position', 'session_id', 'source_group', 'UTC', 'CNT', TARGET]].copy()
audit['cleaning_phase'] = cleaning_phase
audit['training_label'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
audit.loc[clean_mask, 'training_label'] = y.loc[clean_mask]
audit['clean_training_eligible'] = clean_mask
audit.to_csv(OUTPUT_DIR / 'cleaning_audit.csv', index=False)

clean_dataset = df.loc[clean_mask, ['source_row_position', 'session_id', 'source_group', 'UTC', 'CNT']].copy()
clean_dataset[FEATURES15] = X.loc[clean_mask, FEATURES15]
clean_dataset['temperature_baseline'] = baseline_temp.loc[clean_mask]
clean_dataset['pm25_baseline'] = baseline_pm25.loc[clean_mask]
clean_dataset['training_label'] = y.loc[clean_mask]
clean_dataset.to_csv(OUTPUT_DIR / 'smoke_transient_clean_features_v2.csv', index=False)
baseline_table.to_csv(OUTPUT_DIR / 'baseline_values.csv', index=False)
comparison_table.to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
reference_table.to_csv(OUTPUT_DIR / 'reference_0_50.csv', index=False)
group_metrics.to_csv(OUTPUT_DIR / 'group_metrics.csv', index=False)
for name, experiment in experiments.items():
    experiment['sweep'].to_csv(OUTPUT_DIR / f'{name}_threshold_sweep.csv', index=False)
    scores = df.loc[experiment['oof'].index, ['session_id', 'source_group', 'CNT', TARGET]].copy()
    scores['probability'] = experiment['oof']
    if name != 'source13':
        scores['nested_prediction'] = experiment['nested_prediction'].astype(int)
    scores.to_csv(OUTPUT_DIR / f'{name}_oof_scores.csv', index=False)

joblib.dump(final_model, OUTPUT_DIR / 'lightgbm_transient_v2.joblib')
metadata = {
    'model_name': 'LightGBM_Transient_Clean_v2', 'variant': FINAL_VARIANT,
    'features': final_features, 'n_features': len(final_features), 'positive_label': 1,
    'target': 'training_label', 'positive_meaning': 'Sensor-inferred onset window',
    'negative_meaning': 'Selected pre-event background only',
    'baseline': {'method': 'median of first continuous samples, frozen per block',
                 'samples': BASELINE_SAMPLES, 'used_by_exported_model': FINAL_VARIANT == 'clean15',
                 'score_only_after_baseline': FINAL_VARIANT == 'clean15',
                 'requires_known_clean_calibration': FINAL_VARIANT == 'clean15',
                 'comparison_warmup_samples_for_all_variants': BASELINE_SAMPLES},
    'cleaning': {'normal_end_cnt_inclusive': NORMAL_END_CNT,
                 'retain_all_reviewed_onset_windows': True,
                 'excluded_rows_label_in_audit': None},
    'classification_threshold': final_threshold,
    'threshold_selection': selected_experiment['result']['development_selection'],
    'threshold_grid': THRESHOLDS.tolist(),
    'classification_policy': 'instantaneous score comparison; voting configured by consumer',
    'parameters': PARAMETERS,
    'source_sha256': hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
    'physical_fire_ground_truth': 'unverified',
    'versions': {'python': platform.python_version(), 'lightgbm': lgb.__version__,
                 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit_learn': sklearn.__version__},
}
report = {'experiments': {name: v['result'] for name, v in experiments.items()},
          'group_metrics': per_group_rows, 'final_train': final_train_metrics,
          'clean_rows': int(clean_mask.sum()), 'clean_positive': int(y.loc[clean_mask].sum()),
          'clean_negative': int(y.loc[clean_mask].eq(0).sum()),
          'omitted_rows': int((~clean_mask).sum()), 'normal_end_cnt': NORMAL_END_CNT}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
(OUTPUT_DIR / 'results.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
restored = joblib.load(OUTPUT_DIR / 'lightgbm_transient_v2.joblib')
assert list(restored.feature_name_) == final_features
np.testing.assert_allclose(restored.predict_proba(X.loc[final_mask, final_features].head(100)),
                           final_model.predict_proba(X.loc[final_mask, final_features].head(100)))
assert clean_dataset.training_label.isin([0, 1]).all()
assert audit.loc[~audit.clean_training_eligible, 'training_label'].isna().all()
zip_path = OUTPUT_DIR / 'lightgbm_transient_clean_v2_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file() and path != zip_path:
            archive.write(path, arcname=path.name)
print('Bundle:', zip_path)
if FileLink is not None:
    display(FileLink(str(zip_path)))


    threshold  accuracy  precision    recall        f1     TN   FP  FN   TP
0        0.05  0.956203   0.567994  0.908654  0.699029  13457  575  76  756
1        0.06  0.959432   0.589244  0.908654  0.714894  13505  527  76  756
2        0.07  0.961450   0.603352  0.908654  0.725180  13535  497  76  756
3        0.08  0.962998   0.614634  0.908654  0.733269  13558  474  76  756
4        0.09  0.963940   0.621711  0.908654  0.738281  13572  460  76  756
5        0.10  0.965083   0.630525  0.908654  0.744461  13589  443  76  756
6        0.11  0.966025   0.637975  0.908654  0.749628  13603  429  76  756
7        0.12  0.966765   0.643952  0.908654  0.753739  13614  418  76  756
8        0.13  0.967640   0.651163  0.908654  0.758655  13627  405  76  756
9        0.14  0.968447   0.657963  0.908654  0.763251  13639  393  76  756
10       0.15  0.968918   0.661996  0.908654  0.765957  13646  386  76  756
11       0.16  0.969793   0.669619  0.908654  0.771035  13659  373  76  756
12       0.1